# Exploratory analysisSanity checks on the processed panel before any model is run. Answers: coverage, survivorship, liquidity concentration, regime structure.

In [ ]:
import sys, osimport numpy as np, pandas as pdsys.path.insert(0, '../code/data')PROC='../results/processed'ret = pd.read_parquet(f'{PROC}/ret.parquet')bench = pd.read_parquet(f'{PROC}/bench.parquet')liq = pd.read_parquet(f'{PROC}/liq_rank.parquet')uni = pd.read_parquet(f'{PROC}/universe.parquet')print('panel', ret.shape, ret.index[0].date(), ret.index[-1].date())

In [ ]:
# coverage: how many names are eligible over timeelig = (uni.reindex(columns=ret.columns).fillna(False)).sum(axis=1)elig.describe().round(1)

In [ ]:
# survivorship check: names present at the start vs endstart = ret.iloc[:252].notna().sum()end = ret.iloc[-252:].notna().sum()print('names with data in first year :', int((start>200).sum()))print('names with data in last  year :', int((end>200).sum()))print('NOTE: this is current index membership, so names delisted during the period are absent -> survivorship bias, disclosed in the paper')

In [ ]:
# liquidity concentration: how much of the book the top-100 carriestop100 = (liq <= 100)print('median eligible names:', float(elig.median()))print('top-100 share of names:', float(top100.sum(axis=1).median() / elig.median()))

In [ ]:
# regime structure defined in the protocol from the benchmark aloneb = bench['ret']vol = b.rolling(60, min_periods=40).std()*np.sqrt(252)q1,q2 = vol.quantile([1/3,2/3])dd = (1+b.fillna(0)).cumprod(); dd = dd/dd.cummax()-1regimes = pd.Series('mid', index=b.index)regimes[vol<=q1]='low'; regimes[vol>q2]='high'print(regimes.value_counts())print('bull days', int((dd>-0.10).sum()), 'bear days', int((dd<=-0.10).sum()))